## Cleaning the dataset
1. Filtering out URLs that contained query symbols (such as ‘?’ and ‘&’), fragments (#), and some other symbols specific to temporary and script-generated parameters.
The initial (rijksmuseum_archive_urls_full.csv) and resulting (rijksmuseum_archive_urls_cleaned.csv) files are published on Zenodo. DOI: 10.5281/zenodo.21134681

In [ ]:
# cleaning '&', '#', '%', '§', '$', '%', '?', '='

import pandas as pd

# Load the CSV file
file_path = r'data/raw/rijksmuseum_archive_urls_full.csv'
df = pd.read_csv(file_path)

# Filter: only rows NOT containing any of the specific unwanted characters
unwanted_substrings = ['&', '#', '%', '§', '$', '?', '=', '+']
mask = df['original'].apply(lambda x: not any(substr in str(x) for substr in unwanted_substrings))

# Apply mask
df_filtered = df[mask]

# Save result
output_path = r'/data/rijksmuseum_archive_urls_cleaned.csv'
df_filtered.to_csv(output_path, index=False)

print(f"✅ Cleaned data saved. Original rows: {len(df)}, After filtering: {len(df_filtered)}")


2. Deduplication of the URLs (per year)

In [ ]:
# Unique URLs per Year

import pandas as pd
import os

# Input and output paths
input_path = r'data/rijksmuseum_archive_urls_cleaned.csv'
output_path = r'data/Rijksmuseum_unique_urls_per_year.csv"

# Load data
if not os.path.exists(input_path):
    raise FileNotFoundError("❌ File not found.")

df = pd.read_csv(input_path)

# Parse timestamp and extract year
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y%m%d%H%M%S', errors='coerce')
df = df.dropna(subset=['timestamp'])
df['year'] = df['timestamp'].dt.year

# ✅ Keep only one URL per year per digest
df_unique_per_year = df.drop_duplicates(subset=['year', 'digest'])

# Save to a single CSV
df_unique_per_year.to_csv(output_path, index=False)

# Summary
print(f"✅ File with unique URLs per year saved to: {output_path}")
print(f"📊 Total rows: {len(df_unique_per_year)} | Years covered: {df_unique_per_year['year'].nunique()}")